# 1. Perkenalan Dataset

Tahap pertama, eksperimen menggunakan dataset yang berasal dari repositori publik UCI Machine Learning Repository. Dataset dipilih untuk eksperimen klasifikasi dan bukan termasuk Iris, MNIST, Credit Scoring, atau Wine Quality.

**Nama dataset:** Breast Cancer Wisconsin (Diagnostic) Dataset  
**Sumber dataset:** https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic

Dataset memuat 569 sampel dan 30 fitur numerik yang dihitung dari citra digital aspirasi jarum halus (fine needle aspirate/FNA) massa payudara, untuk memprediksi diagnosis `malignant` atau `benign`. Dalam data scikit-learn, label 0 adalah `malignant` dan label 1 adalah `benign`. Eksperimen ini hanya untuk pembelajaran machine learning dan bukan alat diagnosis medis.

Template meminta sumber dataset publik yang jelas. Dataset UCI tersebut dimuat pada notebook dengan fungsi `load_breast_cancer` dari scikit-learn.

# 2. Import Library

Pada tahap ini, impor pustaka Python yang dibutuhkan untuk memuat dan menganalisis data, membuat visualisasi, membagi data, melakukan imputasi dan standardisasi, serta menyimpan hasil preprocessing sebagai CSV.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 3. Memuat Dataset

Pada tahap ini, dataset dimuat dan beberapa baris awal diperiksa untuk memahami strukturnya. Dataset bersumber dari UCI Machine Learning Repository dan tersedia melalui `load_breast_cancer(as_frame=True)` di scikit-learn. Argumen `as_frame=True` menghasilkan DataFrame pandas. Fitur berisi pengukuran numerik inti sel dari citra FNA, sedangkan target menunjukkan diagnosis malignant atau benign.

In [ ]:
dataset = load_breast_cancer(as_frame=True)
data = dataset.data.copy()
data["target"] = dataset.target
data["diagnosis"] = data["target"].map(dict(enumerate(dataset.target_names)))

print("Ukuran dataset:", data.shape)
print("\nLima baris pertama:")
display(data.head())
print("\nNama fitur:", list(dataset.feature_names))
print("Kelas target:", list(dataset.target_names))

# 4. Exploratory Data Analysis (EDA)

Pada tahap ini, lakukan Exploratory Data Analysis (EDA) untuk memahami karakteristik dataset dan memperoleh wawasan awal. Analisis mencakup struktur, statistik fitur, nilai kosong, duplikasi, distribusi kelas, dan distribusi fitur.

In [ ]:
print("Informasi dataset:")
data.info()
print("\nStatistik deskriptif fitur:")
display(data[dataset.feature_names].describe().T)
print("\nJumlah nilai kosong per kolom:")
print(data.isna().sum())
print("\nJumlah baris duplikat:", data.duplicated().sum())
print("\nJumlah data per diagnosis:")
print(data["diagnosis"].value_counts())

data["diagnosis"].value_counts().reindex(dataset.target_names).plot.bar(
    title="Jumlah Data per Diagnosis", xlabel="Diagnosis", ylabel="Jumlah", rot=0
)
plt.tight_layout()
plt.show()

data[dataset.feature_names].hist(figsize=(18, 15), bins=15)
plt.suptitle("Distribusi Fitur Breast Cancer Wisconsin")
plt.tight_layout()
plt.show()

# 5. Data Preprocessing

Pada tahap ini, data dipersiapkan sebelum analisis lebih lanjut. Duplikasi diperiksa tetapi tidak dihapus otomatis. Fitur numerik yang kosong ditangani dengan imputasi median; fitur kemudian distandardisasi. Target sudah berupa angka 0 dan 1 untuk kelas malignant dan benign, sehingga tidak memerlukan encoding tambahan. Data dibagi secara stratified menjadi data latih dan uji. Imputer dan scaler hanya di-fit pada data latih untuk mencegah kebocoran informasi. Notebook ini juga menyimpan data raw serta data train dan test hasil preprocessing menggunakan `to_csv()` pada folder `data/` relatif terhadap working directory notebook.

In [ ]:
X = data[dataset.feature_names]
y = data["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

imputer = SimpleImputer(strategy="median")
X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)

scaler = StandardScaler()
X_train_preprocessed = scaler.fit_transform(X_train_imputed)
X_test_preprocessed = scaler.transform(X_test_imputed)

train_processed = pd.DataFrame(
    X_train_preprocessed, columns=dataset.feature_names
)
train_processed["target"] = y_train.to_numpy()
train_processed["diagnosis"] = train_processed["target"].map(
    dict(enumerate(dataset.target_names))
)

test_processed = pd.DataFrame(
    X_test_preprocessed, columns=dataset.feature_names
)
test_processed["target"] = y_test.to_numpy()
test_processed["diagnosis"] = test_processed["target"].map(
    dict(enumerate(dataset.target_names))
)

raw_dir = Path("data/raw")
processed_dir = Path("data/processed")
raw_dir.mkdir(parents=True, exist_ok=True)
processed_dir.mkdir(parents=True, exist_ok=True)

raw_path = raw_dir / "breast_cancer_raw.csv"
train_path = processed_dir / "breast_cancer_train_preprocessed.csv"
test_path = processed_dir / "breast_cancer_test_preprocessed.csv"
data.to_csv(raw_path, index=False)
train_processed.to_csv(train_path, index=False)
test_processed.to_csv(test_path, index=False)

print("Ukuran data latih:", X_train_preprocessed.shape)
print("Ukuran data uji:", X_test_preprocessed.shape)
print("Jumlah nilai kosong setelah imputasi:", pd.isna(X_train_preprocessed).sum())
print("Label target:", dict(enumerate(dataset.target_names)))
print("Data raw tersimpan di:", raw_path)
print("Data latih tersimpan di:", train_path)
print("Data uji tersimpan di:", test_path)